# EvoSim Deep Dive: Sociological Analysis

This notebook performs a rigorous scrutiny of the EvoSim dataset (N=3000+ agent-years).
We investigate:
1. **The Tribal Imperative**: Is the survival benefit statistically significant?
2. **The Breaking Bad Hypothesis**: Is the negative correlation between Age and Karma robust?
3. **Dimensionality Reduction**: Visualizing the 'Soul Space' via PCA.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import glob

# Load Simulation Data
sim_files = glob.glob('../experiments/*.csv') + glob.glob('../*.csv')
sim_dfs = []
for f in sim_files:
    if 'sim' in f: 
        try: sim_dfs.append(pd.read_csv(f))
        except: pass
full_sim = pd.concat(sim_dfs, ignore_index=True)
print(f"Loaded {len(full_sim)} simulation rows")

## 1. The Tribal Imperative (T-Test)
We test the null hypothesis: *Membership in a tribe has no effect on lifespan.*

In [ ]:
agents = full_sim.sort_values('Tick').groupby('AgentId').last().reset_index()
tribal = agents[agents['TribeId'] != -1]['Age']
lone = agents[agents['TribeId'] == -1]['Age']

t_stat, p_val = stats.ttest_ind(tribal, lone, equal_var=False)
print(f"Tribal Mean Age: {tribal.mean():.2f}")
print(f"Lone Mean Age: {lone.mean():.2f}")
print(f"P-Value: {p_val:.5e}")

sns.boxplot(x=agents['TribeId'] != -1, y=agents['Age'])
plt.xticks([0, 1], ['Lone Wolf', 'Tribal Member'])
plt.title('Survival Advantage of Tribalism')
plt.show()

## 2. The Breaking Bad Hypothesis (Pearson Correlation)
We test the hypothesis: *Age is negatively correlated with Karma.*

In [ ]:
souls_files = [f for f in sim_files if 'soul' in f]
soul_dfs = [pd.read_csv(f) for f in souls_files]
souls = pd.concat(soul_dfs, ignore_index=True)
valid_souls = souls[souls['Age'] > 0]

corr, p_val = stats.pearsonr(valid_souls['Age'], valid_souls['Karma'])
print(f"Correlation: {corr:.3f}")
print(f"P-Value: {p_val:.5e}")

sns.regplot(x='Age', y='Karma', data=valid_souls, scatter_kws={'alpha':0.1}, line_kws={'color':'red'})
plt.title('Age vs Karma: The Breaking Bad Hypothesis')
plt.show()

## 3. Dimensionality Reduction (PCA of Soul Stats)
Can we identify distinct archetypes clustering in 2D space?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

features = ['Altruism', 'Ambition', 'Creativity', 'Karma']
x = souls[features].dropna()
x = StandardScaler().fit_transform(x)

pca = PCA(n_components=2)
principalComponents = pca.fit_transform(x)
principalDf = pd.DataFrame(data = principalComponents, columns = ['PC1', 'PC2'])
finalDf = pd.concat([principalDf, souls[['Archetype']]], axis = 1)

plt.figure(figsize=(10,10))
sns.scatterplot(x='PC1', y='PC2', hue='Archetype', data=finalDf, alpha=0.5)
plt.title('PCA of Soul Archetypes')
plt.show()